# Bus Stoppage & Passenger Shed Detection (YOLOv11)

Section 1: Mount Google Drive

In [ ]:
# Import the drive module from google.colab
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

Section 2: Extract Dataset

In [ ]:
import zipfile
import os

# Define the path to your zip file in Google Drive
zip_path = '/content/drive/MyDrive/Data Science/bus-stop-assess final.v2i.yolov8.zip'
# Define where to extract the dataset
extract_path = '/content/dataset'

# Create the directory if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# Unzip the dataset into the local Colab directory
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraction from Drive completed!")

Section 3: Configure data.yaml

In [ ]:
import yaml

# Path to the dataset configuration file
yaml_path = '/content/dataset/data.yaml'

# Load the existing yaml data
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to match the local Colab environment
data['train'] = '/content/dataset/train/images'
data['val'] = '/content/dataset/valid/images'
data['test'] = '/content/dataset/test/images'

# Save the updated configuration back to the file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Configuration file updated successfully!")

Section 4: Training the Model (Weights & Training)

In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained weights (YOLOv11 Nano)
model = YOLO('yolo11n.pt')

# 2. Start the training process
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/Thesis_Project/training_results',
    name='bus_stop_v1',
    scale=0.5,           # Augmentation: Scaling
    perspective=0.0005,  # Augmentation: Perspective
    mosaic=1.0,          # Augmentation: Mosaic
    mixup=0.2            # Augmentation: Mixup
)

Section 4: Model Training

This section initializes the YOLOv11 model with pre-trained weights and starts the training process with specific augmentations to improve accuracy.

In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLOv11 Nano model
model = YOLO('yolo11n.pt')

# 2. Start the training process
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/Thesis_Project/training_results',
    name='bus_stop_v1',

    # Augmentation parameters to improve model robustness
    scale=0.5,           # Randomly scale images by +/- 50%
    perspective=0.0005,  # Slight perspective change for better angle detection
    mosaic=1.0,          # Mix 4 images into 1 (very effective for small datasets)
    mixup=0.2,           # Blending two images together
    degrees=10.0         # Random rotation within 10 degrees
)

print("Training completed successfully! Weights are saved in your Drive.")

Section 5: Model Inference (Testing on New Images)

After training, we use the best-trained weights (best.pt) to run predictions on the test dataset to visualize how the model performs on unseen data.

In [ ]:
import os
from ultralytics import YOLO

# 1. Load the best weights saved during training
model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
model = YOLO(model_path)

# 2. Define the path for test images
test_images_path = '/content/dataset/test/images'

# 3. Run prediction on the test dataset
if os.path.exists(test_images_path):
    results = model.predict(
        source=test_images_path,
        save=True,          # Save the predicted images with bounding boxes
        conf=0.25,          # Confidence threshold
        project='/content/drive/MyDrive/Thesis_Project',
        name='test_results'
    )
    print("Inference completed! Check the 'test_results' folder in your Drive.")
else:
    print("Error: Test images directory not found.")

Section 6: Quantitative Evaluation (Final Metrics)

This section calculates the final performance metrics (mAP, Precision, Recall) using the test split of your dataset for the thesis report.

In [ ]:
from ultralytics import YOLO

# 1. Load the best-trained model
model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
model = YOLO(model_path)

# 2. Run validation on the test split
metrics = model.val(data='/content/dataset/data.yaml', split='test')

# 3. Print the performance metrics for the thesis report
print("-" * 30)
print(f"Final Results for Thesis:")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")
print("-" * 30)